In [41]:
import sklearn 

In [42]:
import pandas as pd


def load_dataframe():
  path = "high_popularity_spotify_data.csv"
  df = pd.read_csv(path)
  return df

def preprocess_data(df):
  df = df.drop_duplicates(subset=['track_name','track_artist']) #deletes all row duplicates
  df = df.drop(['speechiness','instrumentalness','mode','duration_ms', 'acousticness','track_album_name','track_album_id','playlist_genre','time_signature','track_popularity','playlist_name','track_album_release_date','playlist_id','type', 'playlist_subgenre','analysis_url','track_href','uri','track_id','id'], axis=1) #delete the columns speechiness, liveness, artwork_url and mode
  df = df[~(df == -1).any(axis=1)] #deletes all entries with values that are -1 (not existent)

  # rename track_id to spotify_id
  #df = df.rename(columns={'track_id': 'spotify_id'})
  
  # generate a new column track_id with values from 0 to len(dataframe)
  df['track_id'] = range(len(df))

  # set the track_id as the DataFrame index to standardize indexing
  df.set_index('track_id', inplace=True)
  return df


df = load_dataframe()
df = preprocess_data(df)
df = df[['danceability', 'energy', 'tempo', 'loudness', 'liveness', 'valence', 'key', 'track_name', 'track_artist']]

In [43]:
print(df.head())

          danceability  energy    tempo  loudness  liveness  valence  key  \
track_id                                                                    
0                0.521   0.592  157.969    -7.777     0.122    0.535    6   
1                0.747   0.507  104.978   -10.171     0.117    0.438    2   
2                0.554   0.808  108.548    -4.169     0.159    0.372    1   
3                0.670   0.910  112.966    -4.070     0.304    0.786    0   
4                0.777   0.783  149.027    -4.477     0.355    0.939    0   

                  track_name           track_artist  
track_id                                             
0           Die With A Smile  Lady Gaga, Bruno Mars  
1         BIRDS OF A FEATHER          Billie Eilish  
2             That’s So True          Gracie Abrams  
3                      Taste      Sabrina Carpenter  
4                       APT.       ROSÉ, Bruno Mars  


In [44]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
scalar = StandardScaler()

# Split the data into training and testing sets
train, test = sklearn.model_selection.train_test_split(df, test_size=0.2, random_state=42)
x_train = scalar.fit_transform(train.drop(['track_name', 'track_artist','danceability'], axis=1))
x_test = scalar.transform(test.drop(['track_name', 'track_artist','danceability'], axis=1))
y_train = train['danceability']
y_test = test['danceability']

#model
model = MLPRegressor(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=1000, random_state=42)

#train the model
model.fit(x_train, y_train)
y_pred = model.predict(x_test)



In [45]:
#Evaluation
from sklearn.metrics import r2_score, mean_squared_error
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f'R2 ={r2}')
print(f'MSE ={mse}')

R2 =0.2728861698408057
MSE =0.01761845645631567


In [46]:
for y_pred, y_test in zip(y_pred, y_test):
    print(y_pred - y_test)

0.15934422006338655
-0.06720878370808736
-0.11294360489684718
0.07088110912667467
-0.030530861902380413
-0.10093537815110565
-0.0997688724410224
0.14778731377579984
-0.09654618978213947
0.08729443473151188
0.09149573728508942
-0.04079031907919917
0.17778326286558066
-0.0924900784308853
0.13791364984243337
-0.29860212828249477
0.14300063575122157
0.2967423319365266
0.1659052953246724
-0.06835061786954266
-0.22355103644177332
-0.012906680170785556
-0.05137973017348474
0.02593292151492288
-0.14521187633556576
0.2387959195846205
-0.11361175370842846
0.2183356244279575
0.12935226943830136
0.22228629240027775
0.1797015852223497
-0.14349998433732825
-0.006643505806112726
0.01518484288868005
-0.046681932540659
-0.20000627522737513
-0.22233060500916713
-0.03318968788110932
-0.06408417491790486
-0.10909949632248417
0.06102197348367533
0.00032483093571455957
0.002028278670654471
-0.09661457421522546
-0.21998063833548853
0.026589744984366948
-0.09715186084950023
0.03995841144310708
-0.206253639691